# ML Notebook: PhonePe Pulse Prediction Models


## Sections 1-4: Data Loading & Preprocessing
(Refer to EDA Notebook for detailed exploration)


In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from config import DB_HOST, DB_USER, DB_PASSWORD, DB_NAME
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

# Load Data
db_url = f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
engine = create_engine(db_url)
df = pd.read_sql("SELECT * FROM Aggregated_transaction", con=engine)
df.head()


## Section 5 – Feature Engineering
Encode categorical columns, scale numerical features, define target variable, train-test split (80/20)


In [ ]:
# Feature Engineering
# Target: predict transaction_amount
df = df.dropna()

# Label Encoding
le_state = LabelEncoder()
df['State_Encoded'] = le_state.fit_transform(df['State'])

le_type = LabelEncoder()
df['Type_Encoded'] = le_type.fit_transform(df['transaction_type'])

X = df[['State_Encoded', 'Year', 'Quarter', 'Type_Encoded', 'transaction_count']]
y = df['transaction_amount']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Section 6 – ML Model Training
Train Random Forest, XGBoost, Linear Regression


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=50),
    'XGBoost': XGBRegressor(random_state=42, n_estimators=50)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f"{name} -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}")


In [ ]:
# Evaluation Metric Score Chart
res_df = pd.DataFrame(results).T
res_df[['R2']].plot(kind='bar', title='R2 Score Comparison')
plt.show()


## Section 7 – Cross-Validation & Hyperparameter Tuning
Apply GridSearchCV to Random Forest


In [ ]:
# Hyperparameter tuning for Random Forest
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, None]
}
grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3, scoring='r2')
grid_search.fit(X_train_scaled, y_train)

best_rf = grid_search.best_estimator_
print("Best Params:", grid_search.best_params_)

preds_cv = best_rf.predict(X_test_scaled)
print("Tuned RF R2:", r2_score(y_test, preds_cv))


## Section 8 – Model Comparison & Selection
Explain chosen model and show feature importance


In [ ]:
print("Model Comparison Table:")
display(res_df)

# Feature Importance (Random Forest)
importances = best_rf.feature_importances_
plt.bar(X.columns, importances)
plt.title("Feature Importance")
plt.xticks(rotation=45)
plt.show()


**Model Selected:** Random Forest / XGBoost generally perform the best on tabular data capturing non-linear relationships. We will save the tuned Random Forest model.


## Section 9 – Save & Reload
Save best model with joblib.dump(), reload it and predict on 5 unseen rows


In [ ]:
# Save Model & Scaler
joblib.dump(best_rf, 'best_rf_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Reload and Predict
loaded_model = joblib.load('best_rf_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')

# Predict on 5 unseen rows
sample_X = X_test.head(5)
sample_y = y_test.head(5)
sample_scaled = loaded_scaler.transform(sample_X)

predictions = loaded_model.predict(sample_scaled)
print("Actual values:\n", sample_y.values)
print("Predictions:\n", predictions)


## Conclusion
The machine learning pipeline successfully established models to predict transaction amounts based on geographic and temporal features.
